# SPK-5 — Join Strategies (broadcast vs sort-merge vs shuffle-hash)

**Break → Detect → Fix → Prove.** The *same* big⋈small join can run three different ways. Spark
chooses the **physical join operator** for you — and the wrong choice means a needless full
shuffle of a huge fact (slow), or broadcasting a too-large side that crushes the driver.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch the **SQL / DataFrame** tab.

**The key tool here is `df.explain()`** — it prints the physical plan, names the join operator,
and shows the `Exchange` (shuffle) nodes. It is Spark-Connect-safe (unlike `spark.sparkContext`).

**Laptop-safe:** the fact (~15M rows) is *generated lazily* and only `count()`-ed — never
collected or written — so nothing fills memory or disk. The default **tuned** box is fine
(no need for `make up-constrained`). Nothing to delete at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md) (SQL/DataFrame + Environment tabs), and the
[troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys, key_dimension
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — the join operator shows up in the SQL / DataFrame tab.
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — Parameters & the datasets

A large `events` fact (~15M rows, key spread evenly over `N_KEYS`) joined to a **small**
`dimension` covering those keys. Both are lazy — nothing is stored until an action runs.

In [ ]:
# Lower N_ROWS to 5-10M if your laptop is slow; the strategy/plan is identical at any size.
N_ROWS = 15_000_000
N_KEYS = 2_000      # the dimension is tiny (~2k rows) -> normally broadcastable

fact = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_KEYS, key_col="key")
dim  = key_dimension(spark, n_keys=N_KEYS, key_col="key")

print(f"fact: ~{N_ROWS:,} events (lazy)    dim: {N_KEYS:,} rows (small)")

In [ ]:
# The dimension is small -> Spark would normally broadcast it. (small result, safe to show)
dim.show(5, truncate=False)
print("dim row count:", dim.count())

## 1. Broadcast hash join — the cheapest big⋈small

Under the **`tuned`** profile, `autoBroadcastJoinThreshold = 10 MB`. The ~2k-row dimension is
well under that, so Spark ships it to every executor and streams the fact through — **no shuffle
of the 15M-row fact at all**.

Read the plan below: the join node is **`BroadcastHashJoin`** and the only exchange is a
**`BroadcastExchange`** on the small side.

In [ ]:
apply_profile(spark, "tuned")   # broadcast enabled (10 MB threshold)

bcast_join = fact.join(dim, on="key", how="inner")
bcast_join.explain()            # <- look for 'BroadcastHashJoin' + 'BroadcastExchange'

In [ ]:
m_bcast = measure(spark, "broadcast (BHJ)", lambda: bcast_join.count())
print("\nbroadcast-join metrics:", m_bcast)

## 2. Sort-merge join — broadcast off forces a full shuffle

Now disable broadcasts with `autoBroadcastJoinThreshold = -1` (the **`constrained`** profile does
exactly this). Spark can no longer ship the dimension, so it falls back to a **sort-merge join**
that shuffles **both** sides by `key` and sorts each partition — including all 15M fact rows.

In the plan: the join node is **`SortMergeJoin`**, with **two `Exchange (hashpartitioning…)`**
nodes and a **`Sort`** on each side. That second exchange on the fact is the wasted work.

In [ ]:
apply_profile(spark, "constrained")   # autoBroadcastJoinThreshold = -1 -> no broadcast
profile_summary(spark)

smj_join = fact.join(dim, on="key", how="inner")
smj_join.explain()                    # <- 'SortMergeJoin' + TWO 'Exchange' + 'Sort'

In [ ]:
m_smj = measure(spark, "sort-merge (SMJ)", lambda: smj_join.count())
print("\nsort-merge-join metrics:", m_smj)

## 3. Shuffle-hash join — both sides shuffled, no sort

Spark also has a **shuffle-hash join**: shuffle both sides, then build an in-memory hash map from
the *smaller* side per partition — no sort. It's normally skipped because
`spark.sql.join.preferSortMergeJoin = true`. We flip that off **and** keep broadcast off so the
planner reaches for it (you can also force it with the `.hint("shuffle_hash")` join hint).

In the plan: **`ShuffleHashJoin`**, two `Exchange` nodes, but **no `Sort`** — that's the
difference from sort-merge.

**When Spark picks it on its own:** broadcast is unavailable (side over the threshold) *and* the
smaller side still fits a per-partition hash map (≤ threshold × `shuffleHashJoinFactor`) *and*
`preferSortMergeJoin = false`.

In [ ]:
apply_profile(spark, "constrained", **{
    "spark.sql.join.preferSortMergeJoin": "false",   # allow shuffle-hash to be chosen
})

# A broadcast hint would override everything, so use a shuffle_hash hint to be explicit:
shj_join = fact.join(dim.hint("shuffle_hash"), on="key", how="inner")
shj_join.explain()                                   # <- 'ShuffleHashJoin', two Exchange, NO Sort

In [ ]:
m_shj = measure(spark, "shuffle-hash (SHJ)", lambda: shj_join.count())
print("\nshuffle-hash-join metrics:", m_shj)

## 4. Detect it — read the plan & the Spark UI

The operator name in `df.explain()` above **is** the strategy. Corroborate in the UI
(see [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)):

- **SQL / DataFrame** tab → click the query → the physical-plan DAG shows the same join operator
  and `Exchange` nodes, annotated with per-node rows + **data size**. *"The wrong operator here is
  the bug."*
- **Environment** tab → confirm `spark.sql.autoBroadcastJoinThreshold`. `-1` disables broadcast and
  forces sort-merge; the default 10 MB broadcasts small dimensions — one conf flips the operator.

| In the plan | Strategy | Fact shuffled? |
|-------------|----------|----------------|
| `BroadcastHashJoin` + `BroadcastExchange` (small side) | broadcast | **no** |
| `SortMergeJoin` + two `Exchange` + `Sort` | sort-merge | **yes** |
| `ShuffleHashJoin` + two `Exchange`, no `Sort` | shuffle-hash | yes |

## 5. Fix it — choose deliberately (hints & threshold)

You don't have to accept the planner's estimate. Two production levers, both visible in the plan:

- **Force broadcast** when the side genuinely fits: `F.broadcast(dim)` or the SQL hint
  `/*+ BROADCAST(d) */` (DataFrame: `.hint("broadcast")`).
- **Set the threshold deliberately**: raise it to cover your dimensions, lower it (not all the way
  to `-1`) to stop a borderline fact from being broadcast.

⚠️ **Cost of getting it wrong:** broadcasting a *too-large* side collects it to the **driver** first
→ memory pressure / OOM. That's the **SPK-3** driver-OOM failure mode (*forward reference*) —
broadcast only what fits.

In [ ]:
# Explicit broadcast hint, even with the threshold at -1: the hint wins over the size estimate.
apply_profile(spark, "constrained")        # broadcast OFF by threshold...
hinted = fact.join(F.broadcast(dim), on="key", how="inner")   # ...but the hint forces it back on
hinted.explain()                            # <- 'BroadcastHashJoin' again, despite threshold = -1

# Equivalent via SQL hint syntax:
fact.createOrReplaceTempView("events")
dim.createOrReplaceTempView("dimension")
spark.sql(
    "SELECT /*+ BROADCAST(d) */ e.key, d.tier "
    "FROM events e JOIN dimension d ON e.key = d.key"
).explain()

## 6. Prove it

Before/after across the three strategies. The headline rows for **this** module are **Wall-clock
runtime** and **Shuffle read/write** (not the skew ratio — that's SPK-1's headline). Watch the
broadcast column carry **~0 shuffle on the fact**, while sort-merge and shuffle-hash both pay to
shuffle all 15M rows.

In [ ]:
compare([m_smj, m_shj, m_bcast])

## Takeaways & "when to use which"

- **Read the operator, don't guess** — `df.explain()` (Connect-safe) or the SQL/DataFrame tab names
  the strategy. `BroadcastHashJoin` vs `SortMergeJoin` vs `ShuffleHashJoin` *is* the decision.
- **`autoBroadcastJoinThreshold` is the master switch.** `-1` forces sort-merge; 10 MB broadcasts
  small dims. Set it deliberately — don't cargo-cult `-1` to "fix" an unrelated OOM.
- **When to use which:**
  - **Broadcast** → big⋈small, small side fits the driver/executors. *Try first.*
  - **Sort-merge** → big⋈big, neither side broadcastable. Safe default; shuffles both.
  - **Shuffle-hash** → medium⋈big, smaller side too big to broadcast but fits a per-partition hash; skips the sort.
- **Broadcast only what fits** — a too-large broadcast pressures the driver (SPK-3). Size the side
  *after* filters.
- **In prod:** keep AQE on (it can flip SMJ→broadcast at runtime when a side turns out small), and
  alert on an unexpected `SortMergeJoin` (two `Exchange` nodes) where a broadcast was intended.

## Teardown

Nothing was written (we only counted generated data), so there is nothing to delete. We drop the
temp views and restore the production-tuned safety nets.

In [ ]:
spark.catalog.dropTempView("events")
spark.catalog.dropTempView("dimension")
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")